# CPCV Energy Model Development

This notebook is now split cleanly:

- `model_configs.py` contains the grids/lists of things to test.
- this notebook contains the modelling process.

The flow is step by step:

1. Load configs.
2. Load features.
3. Generate triple-barrier labels.
4. Join labels to features.
5. Build CPCV splits.
6. Apply feature selection inside each fold.
7. Train models inside each fold.
8. Rank by AUC.
9. Inspect top-5 Sharpe distributions.
10. Fit one final model per ticker.

## 1. Imports

Only imports live here. The actual grid values are in `model_configs.py`.

In [ ]:
import os
import sys
import tempfile
import warnings
from itertools import combinations, product
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", os.path.join(tempfile.gettempdir(), "matplotlib"))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as exc:
    XGBClassifier = None
    XGBOOST_AVAILABLE = False
    print("XGBoost unavailable:", exc)

try:
    import shap
    SHAP_AVAILABLE = True
except Exception as exc:
    shap = None
    SHAP_AVAILABLE = False
    print("SHAP unavailable, so SHAP selectors will be skipped:", exc)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "02_tripple_barrier"))
sys.path.append(str(PROJECT_ROOT / "03_model_development"))

from tripple_barrier import run_triple_barrier_pipeline
import model_configs as cfg

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

## 2. Choose Run Mode

Use smoke mode first. The full grid is much larger.

To edit what gets tested, change `03_model_development/model_configs.py`, not this notebook.

In [ ]:
RUN_FULL_GRID = False
SAVE_OUTPUTS = False

TRAINING_END = pd.Timestamp(cfg.TRAINING_END)
FEATURE_MATRIX_PATH = PROJECT_ROOT / "data" / "features" / "merged_feature_matrix.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "models" / "cpcv_energy"

if RUN_FULL_GRID:
    tickers_to_run = cfg.ENERGY_TICKERS
    barrier_configs = cfg.FULL_BARRIER_CONFIGS
    feature_selection_configs = cfg.FULL_FEATURE_SELECTION_CONFIGS
    model_configs = cfg.FULL_MODEL_CONFIGS
else:
    tickers_to_run = cfg.SMOKE_TICKERS
    barrier_configs = cfg.SMOKE_BARRIER_CONFIGS
    feature_selection_configs = cfg.SMOKE_FEATURE_SELECTION_CONFIGS
    model_configs = cfg.SMOKE_MODEL_CONFIGS

if not SHAP_AVAILABLE:
    feature_selection_configs = [c for c in feature_selection_configs if c["method"] != "shap"]
if not XGBOOST_AVAILABLE:
    model_configs = [c for c in model_configs if c["model_type"] != "xgboost"]

print("Run full grid:", RUN_FULL_GRID)
print("Tickers:", tickers_to_run)
print("Barrier configs:", len(barrier_configs))
print("Feature-selection configs:", len(feature_selection_configs))
print("Model configs:", len(model_configs))
print("Candidates per ticker:", len(barrier_configs) * len(feature_selection_configs) * len(model_configs))

## 3. Inspect Imported Configs

This makes it clear what the notebook is about to test.

In [ ]:
display(pd.DataFrame(barrier_configs).head(10))
display(pd.DataFrame(feature_selection_configs))
display(pd.DataFrame([{k: v for k, v in m.items() if k != "params"} | {"params": str(m["params"])} for m in model_configs]))

## 4. Make Model Objects From Configs

`model_configs.py` stores plain dictionaries. This function turns one dictionary into a sklearn/xgboost model.

In [ ]:
def make_model(model_config):
    model_type = model_config["model_type"]
    params = dict(model_config.get("params", {}))
    params.setdefault("random_state", cfg.RANDOM_STATE)

    if model_type == "logistic":
        return LogisticRegression(**params)
    if model_type == "random_forest":
        return RandomForestClassifier(**params)
    if model_type == "extra_trees":
        return ExtraTreesClassifier(**params)
    if model_type == "hist_gradient_boosting":
        return HistGradientBoostingClassifier(**params)
    if model_type == "mlp":
        return MLPClassifier(**params)
    if model_type == "xgboost":
        if not XGBOOST_AVAILABLE:
            raise RuntimeError("XGBoost is not available.")
        return XGBClassifier(**params)

    raise ValueError(f"Unknown model_type: {model_type}")

# Quick check that every active model config can instantiate.
for model_config in model_configs:
    model = make_model(model_config)
    print(model_config["name"], "->", type(model).__name__)

## 5. Load Features

The feature matrix is filtered to energy tickers and pre-2022 training dates.

In [ ]:
def make_primary_signal_long(primary_signal_path):
    signals = pd.read_csv(primary_signal_path, parse_dates=["date"])
    long = signals.melt(id_vars="date", var_name="instrument", value_name="primary_signal")
    return long


def load_or_build_feature_matrix():
    merged_path = PROJECT_ROOT / "data" / "features" / "merged_feature_matrix.csv"
    technical_path = PROJECT_ROOT / "data" / "features" / "technical_analysis_features.csv"
    ohlcv_path = PROJECT_ROOT / "data" / "raw" / "ohlcv_data.csv"
    primary_path = PROJECT_ROOT / "data" / "raw" / "primary_signals.csv"
    hmm_path = PROJECT_ROOT / "data" / "features" / "hmm" / "latent_regime_predictions.csv"

    if merged_path.exists():
        print("Loading merged feature matrix:", merged_path)
        return pd.read_csv(merged_path, parse_dates=["date"])

    print("merged_feature_matrix.csv not found.")
    print("Building feature matrix from available files:")
    print("- raw OHLCV")
    print("- raw primary signals")
    print("- technical_analysis_features.csv")
    print("- HMM latent regime predictions, if available")

    ohlcv = pd.read_csv(ohlcv_path, parse_dates=["date"])
    primary_long = make_primary_signal_long(primary_path)
    technical = pd.read_csv(technical_path, parse_dates=["date"])

    matrix = (
        ohlcv
        .merge(primary_long, on=["date", "instrument"], how="left")
        .merge(technical, on=["date", "instrument"], how="left")
    )

    if hmm_path.exists():
        hmm = pd.read_csv(hmm_path, parse_dates=["date"])
        hmm = hmm.drop(columns=["primary_signal"], errors="ignore")
        matrix = matrix.merge(hmm, on=["date", "instrument"], how="left")
    else:
        print("HMM latent predictions not found, continuing without HMM features.")

    return matrix


features = load_or_build_feature_matrix()
features = features[features["instrument"].isin(cfg.ENERGY_TICKERS)].copy()
features = features[features["date"] < TRAINING_END].copy()
features = features.sort_values(["instrument", "date"]).reset_index(drop=True)

print("Feature matrix:", features.shape)
display(features.groupby("instrument").agg(rows=("date", "size"), min_date=("date", "min"), max_date=("date", "max")))
features.head()

## 6. Generate Labels And Join To Features

This creates the actual modelling table for one ticker and one triple-barrier config.

In [ ]:
label_cache = {}

leakage_columns = {
    "training_end", "vol", "tp", "sl", "timeout_date", "timeout_close",
    "touch_date", "touch_price", "touched_barrier", "triple_barrier_label",
    "metalabel", "holding_period_days", "raw_touch_return", "signed_touch_return",
    "volatility_method", "ewma_span", "volatility_window", "num_days",
    "take_profit_mult", "stop_loss_mult",
}


def get_labels(ticker, barrier_config):
    key = (ticker, tuple(sorted(barrier_config.items())))
    if key not in label_cache:
        kwargs = {k: v for k, v in barrier_config.items() if k != "name"}
        label_cache[key] = run_triple_barrier_pipeline(
            instrument=ticker,
            training_end=TRAINING_END,
            **kwargs,
        )
    return label_cache[key].copy()


def make_model_data(ticker, barrier_config):
    labels = get_labels(ticker, barrier_config)
    ticker_features = features[features["instrument"] == ticker].copy()

    labels_for_join = labels.drop(columns=["close"], errors="ignore")
    ticker_features = ticker_features.drop(columns=["primary_signal"], errors="ignore")

    data = labels_for_join.merge(ticker_features, on=["date", "instrument"], how="inner")
    data = data.sort_values("date").reset_index(drop=True)

    numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
    feature_cols = [c for c in numeric_cols if c not in leakage_columns]
    data[feature_cols] = data[feature_cols].replace([np.inf, -np.inf], np.nan)
    return data, feature_cols


example_data, example_feature_cols = make_model_data(tickers_to_run[0], barrier_configs[0])
print("Rows:", len(example_data))
print("Feature columns:", len(example_feature_cols))
print("Metalabel counts:")
print(example_data["metalabel"].value_counts().sort_index())
example_data.head()

## 7. CPCV Splits

This is the purged CPCV logic. It is kept as small as possible:

- split events into chronological groups
- choose test group combinations
- remove train rows whose event windows overlap the test window
- embargo a few observations after the test window

In [ ]:
def make_groups(n_rows, n_groups):
    return [g.astype(int) for g in np.array_split(np.arange(n_rows), n_groups) if len(g) > 0]


def embargo_date(dates, end_date, embargo):
    unique_dates = pd.Series(pd.to_datetime(dates).sort_values().unique())
    pos = unique_dates.searchsorted(end_date, side="right")
    if pos >= len(unique_dates):
        return end_date
    return unique_dates.iloc[min(len(unique_dates) - 1, pos + embargo - 1)]


def make_cpcv_splits(data, ticker):
    settings = cfg.CPCV_SETTINGS[ticker] if RUN_FULL_GRID else cfg.SMOKE_CPCV_SETTINGS
    groups = make_groups(len(data), settings["n_groups"])
    splits = []

    for split_id, test_group_ids in enumerate(combinations(range(len(groups)), settings["k_test_groups"])):
        test_idx = np.concatenate([groups[i] for i in test_group_ids])
        train_mask = np.ones(len(data), dtype=bool)
        train_mask[test_idx] = False

        for group_id in test_group_ids:
            group_idx = groups[group_id]
            test_start = data.loc[group_idx, "date"].min()
            test_end = data.loc[group_idx, "date"].max()
            test_horizon_end = data.loc[group_idx, "timeout_date"].max()
            emb_end = embargo_date(data["date"], test_end, settings["embargo"])

            overlapping = (data["date"] <= test_horizon_end) & (data["timeout_date"] >= test_start)
            embargoed = (data["date"] > test_end) & (data["date"] <= emb_end)
            train_mask[overlapping | embargoed] = False

        train_idx = np.where(train_mask)[0]
        if len(train_idx) > 0 and len(test_idx) > 0:
            splits.append({
                "split_id": split_id,
                "test_groups": test_group_ids,
                "train_idx": train_idx,
                "test_idx": test_idx,
                "n_train": len(train_idx),
                "n_test": len(test_idx),
            })
    return splits


example_splits = make_cpcv_splits(example_data, tickers_to_run[0])
print("Number of splits:", len(example_splits))
display(pd.DataFrame([{k: v for k, v in split.items() if k not in ["train_idx", "test_idx"]} for split in example_splits]))

## 8. Feature Selection Functions

Each selector fits only on the train fold.

In [ ]:
def select_none(X_train, X_test, y_train, fs_config):
    return X_train.values, X_test.values, list(X_train.columns), None


def select_mdi(X_train, X_test, y_train, fs_config):
    k = min(fs_config.get("k", 20), X_train.shape[1])
    tree = ExtraTreesClassifier(n_estimators=100, min_samples_leaf=5, class_weight="balanced", random_state=cfg.RANDOM_STATE, n_jobs=-1)
    tree.fit(X_train, y_train)
    importance = pd.Series(tree.feature_importances_, index=X_train.columns).sort_values(ascending=False)
    selected = importance.head(k).index.tolist()
    report = importance.head(k).reset_index().rename(columns={"index": "feature", 0: "importance"})
    return X_train[selected].values, X_test[selected].values, selected, report


def select_shap(X_train, X_test, y_train, fs_config):
    if not SHAP_AVAILABLE:
        raise RuntimeError("SHAP is not available.")
    k = min(fs_config.get("k", 20), X_train.shape[1])
    tree = ExtraTreesClassifier(n_estimators=100, min_samples_leaf=5, class_weight="balanced", random_state=cfg.RANDOM_STATE, n_jobs=-1)
    tree.fit(X_train, y_train)
    sample = X_train.sample(min(200, len(X_train)), random_state=cfg.RANDOM_STATE)
    explainer = shap.TreeExplainer(tree)
    shap_values = explainer.shap_values(sample)
    if isinstance(shap_values, list):
        values = shap_values[1]
    elif getattr(shap_values, "ndim", 0) == 3:
        values = shap_values[:, :, 1]
    else:
        values = shap_values
    importance = pd.Series(np.abs(values).mean(axis=0), index=X_train.columns).sort_values(ascending=False)
    selected = importance.head(k).index.tolist()
    report = importance.head(k).reset_index().rename(columns={"index": "feature", 0: "importance"})
    return X_train[selected].values, X_test[selected].values, selected, report


def select_cluster(X_train, X_test, y_train, fs_config):
    corr_threshold = fs_config.get("corr_threshold", 0.90)
    max_features = fs_config.get("max_features", 50)

    corr = X_train.corr().abs().fillna(0).clip(0, 1)
    distance = (1 - corr).clip(0, 1)
    np.fill_diagonal(distance.values, 0)

    linkage = hierarchy.linkage(squareform(distance.values, checks=False), method="average")
    clusters = hierarchy.fcluster(linkage, t=1 - corr_threshold, criterion="distance")
    mi = pd.Series(mutual_info_classif(X_train, y_train, random_state=cfg.RANDOM_STATE), index=X_train.columns)

    rows = []
    for cluster_id in sorted(np.unique(clusters)):
        members = [X_train.columns[i] for i, c in enumerate(clusters) if c == cluster_id]
        best = mi.loc[members].sort_values(ascending=False).index[0]
        rows.append({"feature": best, "cluster": cluster_id, "score": mi.loc[best], "cluster_size": len(members)})

    report = pd.DataFrame(rows).sort_values("score", ascending=False).head(max_features)
    selected = report["feature"].tolist()
    return X_train[selected].values, X_test[selected].values, selected, report


def select_pca(X_train, X_test, y_train, fs_config):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    pca = PCA(n_components=fs_config.get("variance", 0.90), random_state=cfg.RANDOM_STATE)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    names = [f"PC{i + 1}" for i in range(X_train_pca.shape[1])]
    report = pd.DataFrame({
        "component": names,
        "explained_variance": pca.explained_variance_ratio_,
        "cumulative_explained_variance": np.cumsum(pca.explained_variance_ratio_),
    })
    return X_train_pca, X_test_pca, names, report


def select_cluster_pca(X_train, X_test, y_train, fs_config):
    X_train_cluster, X_test_cluster, selected, cluster_report = select_cluster(X_train, X_test, y_train, fs_config)
    X_train_cluster = pd.DataFrame(X_train_cluster, columns=selected, index=X_train.index)
    X_test_cluster = pd.DataFrame(X_test_cluster, columns=selected, index=X_test.index)
    X_train_pca, X_test_pca, names, pca_report = select_pca(X_train_cluster, X_test_cluster, y_train, fs_config)
    return X_train_pca, X_test_pca, names, {"cluster_report": cluster_report, "pca_report": pca_report}


def apply_feature_selection(X_train, X_test, y_train, fs_config):
    method = fs_config["method"]
    if method == "none":
        return select_none(X_train, X_test, y_train, fs_config)
    if method == "mdi":
        return select_mdi(X_train, X_test, y_train, fs_config)
    if method == "shap":
        return select_shap(X_train, X_test, y_train, fs_config)
    if method == "cluster":
        return select_cluster(X_train, X_test, y_train, fs_config)
    if method == "pca":
        return select_pca(X_train, X_test, y_train, fs_config)
    if method == "cluster_pca":
        return select_cluster_pca(X_train, X_test, y_train, fs_config)
    raise ValueError(f"Unknown feature selection method: {method}")

## 9. Evaluate One Candidate

This trains and scores one candidate across CPCV splits.

In [ ]:
def prediction_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    scores = model.decision_function(X)
    return 1 / (1 + np.exp(-scores))


def sharpe_ratio(returns):
    returns = pd.Series(returns).dropna()
    if len(returns) < 2 or returns.std(ddof=1) == 0:
        return np.nan
    return returns.mean() / returns.std(ddof=1) * np.sqrt(len(returns))


def evaluate_candidate(ticker, barrier_config, fs_config, model_config):
    data, feature_cols = make_model_data(ticker, barrier_config)
    splits = make_cpcv_splits(data, ticker)
    rows = []
    first_report = None

    for split in splits:
        train = data.iloc[split["train_idx"]]
        test = data.iloc[split["test_idx"]]
        y_train = train["metalabel"].astype(int)
        y_test = test["metalabel"].astype(int)

        if y_train.nunique() < 2:
            continue

        X_train_raw = train[feature_cols].replace([np.inf, -np.inf], np.nan)
        X_test_raw = test[feature_cols].replace([np.inf, -np.inf], np.nan)

        imputer = SimpleImputer(strategy="median")
        X_train = pd.DataFrame(imputer.fit_transform(X_train_raw), columns=feature_cols, index=train.index)
        X_test = pd.DataFrame(imputer.transform(X_test_raw), columns=feature_cols, index=test.index)

        X_train_selected, X_test_selected, selected_features, fs_report = apply_feature_selection(X_train, X_test, y_train, fs_config)

        if model_config["needs_scaling"]:
            scaler = StandardScaler()
            X_train_model = scaler.fit_transform(X_train_selected)
            X_test_model = scaler.transform(X_test_selected)
        else:
            X_train_model = X_train_selected
            X_test_model = X_test_selected

        model = make_model(model_config)
        model.fit(X_train_model, y_train)
        y_score = prediction_scores(model, X_test_model)
        y_pred = (y_score >= cfg.PROBA_THRESHOLD).astype(int)

        auc = roc_auc_score(y_test, y_score) if y_test.nunique() == 2 else np.nan
        trade_returns = test.loc[y_pred == 1, "signed_touch_return"]

        rows.append({
            "ticker": ticker,
            "barrier": barrier_config["name"],
            "feature_selection": fs_config["name"],
            "model": model_config["name"],
            "split_id": split["split_id"],
            "auc": auc,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "sharpe": sharpe_ratio(trade_returns),
            "trade_count": int((y_pred == 1).sum()),
            "selected_feature_count": len(selected_features),
        })

        if first_report is None:
            first_report = {
                "ticker": ticker,
                "barrier": barrier_config["name"],
                "feature_selection": fs_config["name"],
                "model": model_config["name"],
                "selected_features": selected_features,
                "report": fs_report,
            }

    return rows, first_report

## 10. Run Grid

This is the main loop. In smoke mode it is intentionally small.

In [ ]:
all_rows = []
feature_selection_reports = []

total = len(tickers_to_run) * len(barrier_configs) * len(feature_selection_configs) * len(model_configs)
print("Total candidates:", total)

count = 0
for ticker, barrier_config, fs_config, model_config in product(tickers_to_run, barrier_configs, feature_selection_configs, model_configs):
    count += 1
    print(f"{count}/{total}: {ticker} | {barrier_config['name']} | {fs_config['name']} | {model_config['name']}")
    rows, report = evaluate_candidate(ticker, barrier_config, fs_config, model_config)
    all_rows.extend(rows)
    if report is not None:
        feature_selection_reports.append(report)

all_cpcv_results = pd.DataFrame(all_rows)
print("Path-level results:", all_cpcv_results.shape)
all_cpcv_results.head()

## 11. Summarise And Rank

Rank candidates by mean CPCV AUC.

In [ ]:
candidate_summary = (
    all_cpcv_results
    .groupby(["ticker", "barrier", "feature_selection", "model"], as_index=False)
    .agg(
        mean_auc=("auc", "mean"),
        median_auc=("auc", "median"),
        valid_auc_paths=("auc", lambda x: x.notna().sum()),
        median_sharpe=("sharpe", "median"),
        mean_sharpe=("sharpe", "mean"),
        sharpe_iqr=("sharpe", lambda x: x.quantile(0.75) - x.quantile(0.25)),
        sharpe_std=("sharpe", "std"),
        total_trades=("trade_count", "sum"),
        avg_selected_features=("selected_feature_count", "mean"),
    )
)

candidate_summary = candidate_summary[candidate_summary["valid_auc_paths"] >= cfg.MIN_VALID_AUC_PATHS].copy()
candidate_summary = candidate_summary.sort_values(["ticker", "mean_auc"], ascending=[True, False]).reset_index(drop=True)

print("Candidate summary:", candidate_summary.shape)
candidate_summary.head(20)

## 12. Top 5 And Sharpe Distributions

Take the top 5 by AUC for each ticker, then look at Sharpe distributions.

In [ ]:
top5_by_ticker = {}

for ticker, group in candidate_summary.groupby("ticker"):
    top5 = group.head(cfg.TOP_K).copy()
    top5_by_ticker[ticker] = top5

    print()
    print(f"Top {len(top5)} candidates for {ticker}")
    display(top5)

    plot_data = []
    labels = []
    for _, row in top5.iterrows():
        mask = (
            (all_cpcv_results["ticker"] == row["ticker"])
            & (all_cpcv_results["barrier"] == row["barrier"])
            & (all_cpcv_results["feature_selection"] == row["feature_selection"])
            & (all_cpcv_results["model"] == row["model"])
        )
        plot_data.append(all_cpcv_results.loc[mask, "sharpe"].dropna().values)
        labels.append(chr(10).join([row["feature_selection"], row["model"]]))

    if plot_data:
        plt.figure(figsize=(12, 5))
        plt.boxplot(plot_data, labels=labels, showmeans=True)
        plt.axhline(0, color="black", linewidth=1, alpha=0.5)
        plt.title(f"{ticker}: Sharpe distribution for top-{len(top5)} AUC configs")
        plt.ylabel("Sharpe across CPCV paths")
        plt.xticks(rotation=20, ha="right")
        plt.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()

## 13. Select Final Configs

Final rule: highest median Sharpe from the top-5 AUC candidates, with tighter Sharpe spread as tie-breaker.

In [ ]:
selected_configs = {}

for ticker, top5 in top5_by_ticker.items():
    ranked = top5.sort_values(
        ["median_sharpe", "sharpe_iqr", "sharpe_std", "mean_auc"],
        ascending=[False, True, True, False],
    ).reset_index(drop=True)
    selected_configs[ticker] = ranked.iloc[0].to_dict()

selected_configs_df = pd.DataFrame(selected_configs).T
selected_configs_df

## 14. Inspect Selected Feature Reports

This shows the selected feature list or feature-importance report for each final config.

In [ ]:
def find_feature_report(selected):
    for report in feature_selection_reports:
        if (
            report["ticker"] == selected["ticker"]
            and report["barrier"] == selected["barrier"]
            and report["feature_selection"] == selected["feature_selection"]
            and report["model"] == selected["model"]
        ):
            return report
    return None

for ticker, selected in selected_configs.items():
    print()
    print("Selected config for", ticker)
    report = find_feature_report(selected)
    if report is None:
        print("No feature report found.")
        continue
    print("Feature selection:", report["feature_selection"])
    print("Selected feature count:", len(report["selected_features"]))
    if isinstance(report["report"], pd.DataFrame):
        display(report["report"].head(30))
    else:
        print(report["selected_features"][:30])

## 15. Fit Final Models

Fit one final model per selected ticker/config on all available pre-2022 labelled rows.

In [ ]:
def get_config_by_name(configs, name):
    for config in configs:
        if config["name"] == name:
            return config
    raise KeyError(name)


def fit_final_model(selected):
    ticker = selected["ticker"]
    barrier_config = get_config_by_name(barrier_configs, selected["barrier"])
    fs_config = get_config_by_name(feature_selection_configs, selected["feature_selection"])
    model_config = get_config_by_name(model_configs, selected["model"])

    data, feature_cols = make_model_data(ticker, barrier_config)
    y = data["metalabel"].astype(int)
    X_raw = data[feature_cols].replace([np.inf, -np.inf], np.nan)

    imputer = SimpleImputer(strategy="median")
    X = pd.DataFrame(imputer.fit_transform(X_raw), columns=feature_cols, index=data.index)
    X_selected, _, selected_features, fs_report = apply_feature_selection(X, X, y, fs_config)

    scaler = None
    if model_config["needs_scaling"]:
        scaler = StandardScaler()
        X_model = scaler.fit_transform(X_selected)
    else:
        X_model = X_selected

    model = make_model(model_config)
    model.fit(X_model, y)

    return {
        "ticker": ticker,
        "barrier_config": barrier_config,
        "feature_selection_config": fs_config,
        "model_config": model_config,
        "imputer": imputer,
        "scaler": scaler,
        "selected_features": selected_features,
        "feature_selection_report": fs_report,
        "model": model,
        "training_rows": len(data),
        "feature_cols_before_selection": feature_cols,
    }

final_models = {ticker: fit_final_model(selected) for ticker, selected in selected_configs.items()}

pd.DataFrame([
    {
        "ticker": ticker,
        "model": obj["model_config"]["name"],
        "feature_selection": obj["feature_selection_config"]["name"],
        "training_rows": obj["training_rows"],
        "selected_features": len(obj["selected_features"]),
    }
    for ticker, obj in final_models.items()
])

## 16. Optional Save

Nothing is saved unless `SAVE_OUTPUTS = True`.

In [ ]:
if SAVE_OUTPUTS:
    import joblib

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    all_cpcv_results.to_csv(OUTPUT_DIR / "path_level_results.csv", index=False)
    candidate_summary.to_csv(OUTPUT_DIR / "candidate_summary.csv", index=False)
    selected_configs_df.to_csv(OUTPUT_DIR / "selected_configs.csv")

    for ticker, obj in final_models.items():
        joblib.dump(obj, OUTPUT_DIR / f"{ticker}_final_model.joblib")

    print("Saved outputs to", OUTPUT_DIR)
else:
    print("SAVE_OUTPUTS is False, so nothing was written.")